# Partie 8 : Vers une Ontologie du Gameplay - La Linguistique du Design

## Objectif et Perspective
Après avoir exploré l'**Usage Social** (Partie 3 : associations horizontales) et la **Stabilisation Sémantique** (Partie 7 : plongements vectoriels), ce dernier volet s'attache à la **Linguistique du Design**. Nous passons ici d'une vision "à plat" de la folksonomie à une structure verticale et évolutive.

L'objectif est de transformer nos clusters en une véritable ontologie du gameplay en mesurant deux phénomènes critiques :
1. **Hiérarchisation (Subsumption)** : Identifier les relations Parent-Enfant (Sanderson & Croft) pour comprendre comment les concepts généraux (ex: Action) engendrent des spécialisations (ex: Tactical RPG).
2. **Complexité Sémantique (Hybridité)** : Utiliser l'**Entropie de Shannon** pour quantifier la richesse et le mélange des genres au sein d'un même titre.
3. **Dérive Sémantique (Diachronie)** : Analyser comment le sens et la force d'association des termes évoluent au fil du temps (ex: Roguelike vs Roguelite), marquant la maturation du langage ludique.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
plt.rcParams.update({
    'figure.facecolor': 'white', 
    'axes.facecolor': 'white', 
    'savefig.facecolor': 'white',
    'axes.edgecolor': 'black',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'text.color': 'black',
    'figure.autolayout': True
})
def save_plot(name):
    if not os.path.exists('figures'): os.makedirs('figures')
    plt.gcf().canvas.draw()
    plt.savefig(f'figures/8_{name}.png', bbox_inches='tight', facecolor='white', dpi=300)
import seaborn as sns
import networkx as nx
import os
import math

df = pd.read_csv('../data/Games_Gameplay_Taxonomy.csv')
df['game_release_date'] = pd.to_datetime(df['game_release_date'].astype(str).str.zfill(6), format='%y%m%d', errors='coerce')
df['year'] = df['game_release_date'].dt.year

print(f"Base chargée : {len(df)} jeux.")

## 1. Algorithme de Hiérarchisation (Subsumption)

Nous utilisons la règle de subsumption de **Sanderson & Croft (1999)**. 
Un tag $A$ subsume (est le parent de) $B$ si :
- $P(A|B) \geq 0.8$
- $P(B|A) < P(A|B)$

In [ ]:
def calculate_subsumption(df, tags_list, threshold=0.8):
    tag_data = {}
    for tag in tags_list:
        tag_data[tag] = df['Genre'].str.contains(tag, na=False) | df['Mechanics'].str.contains(tag, na=False)
    
    tag_df = pd.DataFrame(tag_data)
    counts = tag_df.sum()
    hierarchy = []
    tags = tag_df.columns
    
    for i in range(len(tags)):
        for j in range(len(tags)):
            if i == j: continue
            A, B = tags[i], tags[j]
            intersection = (tag_df[A] & tag_df[B]).sum()
            p_a_given_b = intersection / counts[B] if counts[B] > 0 else 0
            p_b_given_a = intersection / counts[A] if counts[A] > 0 else 0
            
            if p_a_given_b >= threshold and p_a_given_b > p_b_given_a:
                hierarchy.append({'Parent': A, 'Child': B, 'Confidence': p_a_given_b})
                
    return pd.DataFrame(hierarchy)

top_tags = [
    'Action', 'Shooter', 'FPS', 'Third-Person Shooter', 'Arena Shooter', 'Bullet Hell', 'Shoot \'Em Up',
    'Platformer', '2D Platformer', '3D Platformer', 'Metroidvania', 'Precision Platformer',
    'RPG', 'Action RPG', 'JRPG', 'CRPG', 'Tactical RPG', 'Roguelike', 'Roguelite', 'Action Roguelike',
    'Strategy', 'Turn-Based Strategy', 'RTS', 'Tower Defense', 'Grand Strategy', '4X', 'Real Time Tactics',
    'Simulation', 'Management', 'City Builder', 'Colony Sim', 'Life Sim', 'Farming Sim',
    'Adventure', 'Visual Novel', 'Puzzle', 'Point & Click', 'Interactive Fiction',
    'Horror', 'Survival Horror', 'Psychological Horror'
]
df_hierarchy = calculate_subsumption(df, top_tags)
print("Relations hiérarchiques détectées :")
print(df_hierarchy.sort_values('Confidence', ascending=False))

G = nx.DiGraph()
for _, row in df_hierarchy.iterrows():
    G.add_edge(row['Parent'], row['Child'], weight=row['Confidence'])

plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=1.5)
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=3000, font_size=10, arrowsize=20)
plt.title("Ontologie extraite de la Folksonomie")
plt.show()


### Analyse : Hiérarchie de Subsumption
L'ontologie ainsi créée montre une structure verticale claire :
- **Généricité vs Spécificité** : Les tags "Parents" (Action, RPG) sont sémantiquement pauvres car trop inclusifs, tandis que les "Enfants" portent la richesse descriptive.
- **Validation** : Le fait que $P(Action|Shooter)$ soit élevé valide notre approche statistique de la hiérarchie parent-enfant sans intervention humaine.


*On remarque que de nombreux jeux sont en effet des enfants issue de genres plus larges. Les genres isuuent de fusion de genre en sont l'exemple le plus notable, 'Action Roguelike', 'Third-Person Shooter' ou 'JRPG' sont des enfants de 'Action', 'Shooter' et 'RPG' respectivement.\
Cependant, certains genres comme 'Roguelite' ne sont pas clairement subsumés par 'Roguelike', ce qui suggère une divergence sémantique potentielle. Cette observation nous conduit à l'analyse diachronique pour mesurer cette divergence au fil du temps.*

## 2. Analyse Diachronique : Spéciation Sémantique (PMI Drift)

Selon **Hamilton et al. (2016)**, le changement sémantique peut être mesuré par l'évolution de la force d'association. Nous utilisons le **Pointwise Mutual Information (PMI)** pour observer si 'Roguelike' et 'Roguelite' deviennent indépendants.
Un PMI élevé indique une forte dépendance (synonymie), un PMI proche de 0 indique une indépendance sémantique.

In [ ]:
evol_data = []
N_total = len(df)

for year in range(2010, 2026):
    year_df = df[df['year'] == year]
    n_year = len(year_df)
    if n_year < 50: continue
    
    p_like = year_df['Genre'].str.contains('Roguelike', na=False).sum() / n_year
    p_lite = year_df['Genre'].str.contains('Roguelite', na=False).sum() / n_year
    p_both = (year_df['Genre'].str.contains('Roguelike', na=False) & year_df['Genre'].str.contains('Roguelite', na=False)).sum() / n_year
    
    pmi = np.log2(p_both / (p_like * p_lite)) if p_both > 0 else 0
    evol_data.append({'Year': year, 'PMI': pmi, 'Roguelike_Freq': p_like, 'Roguelite_Freq': p_lite})

df_evol = pd.DataFrame(evol_data)

fig, ax1 = plt.subplots(figsize=(12, 6))
sns.lineplot(data=df_evol, x='Year', y='PMI', marker='o', color='purple', label='Force d association (PMI)', ax=ax1)
ax1.axhline(y=0, color='black', linestyle='--')
ax1.set_ylabel("PMI (Indépendance < 0 > Dépendance)")

ax2 = ax1.twinx()
ax2.bar(df_evol['Year'], df_evol['Roguelite_Freq'] * 100, alpha=0.3, label='Fréquence Roguelite (%)', color='red')
ax2.set_ylabel('% de jeux')

plt.title("Analyse de la Dérive Sémantique : Roguelike vs Roguelite")
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
save_plot('plot')
plt.show()

*On peut voir que au fur et à mesure des années, le PMI entre 'Roguelike' et 'Roguelite' diminue, indiquant une divergence sémantique. Cela suggère que 'Roguelite' est de plus en plus perçu comme un genre distinct, confirmant la spéciation sémantique. La diminution du PMI est corrélée à l'augmentation de la fréquence de 'Roguelite', renforçant l'idée d'une émergence d'un nouveau genre.\
Cette analyse diachronique met en lumière la dynamique évolutive des genres de jeux vidéo, illustrant comment les catégories peuvent se différencier au fil du temps en réponse à l'innovation et à la diversification des mécaniques de gameplay.*

## 3. Complexité Sémantique : Entropie de Shannon et Hybridité

S'inspirant de **Hsu (2006)** sur la sociologie des catégories, nous calculons l'**Entropie de Shannon** ($H$) pour chaque jeu. 
$H = -\sum p_i \log p_i$ où $p_i$ est la proportion de tags du jeu appartenant au cluster $i$.
Une entropie élevée indique un jeu dont les mécaniques sont distribuées sur plusieurs super-genres (véritable hybridité).

In [ ]:
try:
    df_clusters = pd.read_csv('../data/Folksonomic_Clusters.csv')
    tag_to_cid = dict(zip(df_clusters['Tag'], df_clusters['Cluster']))
    
    def calculate_game_entropy(row):
        tags = []
        for col in ['Genre', 'Mechanics']:
            if pd.notna(row[col]):
                tags.extend([t.strip() for t in row[col].split(',')])
        
        cids = [tag_to_cid.get(t) for t in tags if t in tag_to_cid]
        if not cids: return 0
        
        counts = pd.Series(cids).value_counts()
        probs = counts / len(cids)
        return -sum(p * math.log2(p) for p in probs)
    
    df['Shannon_Entropy'] = df.apply(calculate_game_entropy, axis=1)
    
    plt.figure(figsize=(10, 6))
    sns.histplot(df['Shannon_Entropy'], bins=30, kde=True, color='teal')
    plt.title("Distribution de l'Hybridité (Entropie de Shannon)")
    plt.xlabel("Indice d'Entropie (0 = Pur, >1 = Hybride)")
    plt.show()
    
    print(f"Entropie moyenne : {df['Shannon_Entropy'].mean():.2f}")
    print("\nTop 10 jeux les plus 'hybrides' sémantiquement :")
    print(df.sort_values('Shannon_Entropy', ascending=False)[['game_name', 'Shannon_Entropy']].head(10))
except FileNotFoundError:
    print("Veuillez d'abord générer le fichier Folksonomic_Clusters.csv (Notebook 4).")

*La plupart des jeux ont une entropie relativement faible, indiquant qu'ils sont principalement associés à un ou deux clusters (genres). Cependant, certains jeux présentent une entropie élevée, suggérant une hybridité sémantique importante. L'entropie moyenne est 0.95, ce qui indique que de nombreux jeux combinent des éléments de plusieurs genres, reflétant la tendance actuelle à la fusion et à l'innovation dans le design de jeux vidéo.*

## Conclusion : Vers une Compréhension Dynamique des Genres

Cette analyse finale a permis de dépasser la vision statique des taxonomies de jeux vidéo pour embrasser leur nature évolutive :

1. **Hiérarchisation** : L'application de l'algorithme de subsumption a révélé une structure logique où les genres "piliers" (Action, RPG, Strategy) agissent comme des parents sémantiques pour une myriade de sous-genres spécialisés. Cette hiérarchie confirme que la folksonomie Steam, malgré son bruit, contient une structure ontologique cohérente.
2. **Diachronie et Spéciation** : L'étude du PMI (Pointwise Mutual Information) entre *Roguelike* et *Roguelite* constitue une preuve statistique de la "spéciation sémantique". Au fil du temps, ces deux termes se sont éloignés l'un de l'autre pour désigner des expériences distinctes, illustrant comment le langage des joueurs s'adapte à l'innovation du design.
3. **Hybridité et Complexité** : L'utilisation de l'entropie de Shannon a permis de quantifier l'hybridité des jeux. L'entropie moyenne de 0.95 suggère que le paysage ludique moderne est intrinsèquement multi-facettes, rendant les classifications monolithiques de plus en plus obsolètes.

En conclusion, ce projet démontre que les tags utilisateurs ne sont pas seulement des outils de recherche, mais une mine d'or pour la recherche ludologique, permettant de cartographier en temps réel l'évolution d'une industrie en perpétuelle mutation.